# 11 PHA Enzyme Docking

This notebook prepares benchmark PHA oligomer structures for manual HADDOCK docking. It starts from the final GROMACS production structure `step7_production.gro` and exports a docking-ready polymer PDB.

Scope: locate benchmark structures, export docking PDBs, document enzyme/polymer docking jobs, and prepare files for manual HADDOCK runs. This notebook does not submit HADDOCK jobs, does not add DiffDock or RFdiffusion, and does not start enzyme-polymer MD.

## Select Polymer System

Change `polymer_system` to prepare a different benchmark oligomer.

In [1]:
supported_systems = [
    "PHB4",
    "PHB8",
    "PHO4",
    "PHO8",
    "PHDD4",
    "PHDD8",
]

polymer_system = "PHB4"
if polymer_system not in supported_systems:
    raise ValueError(f"Unsupported polymer system: {polymer_system}")

polymer_system

'PHB4'

## Locate Benchmark Structure

The expected final MD structure is:

`examples/output/md_tests/benchmark/{polymer_system}/gromacs/solvated_polymer/step7_production.gro`

In [2]:
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
benchmark_root = repo_root / "examples" / "output" / "md_tests" / "benchmark"
solvated_polymer_dir = benchmark_root / polymer_system / "gromacs" / "solvated_polymer"
final_gro_path = solvated_polymer_dir / "step7_production.gro"
docking_output_dir = repo_root / "examples" / "output" / "docking_inputs" / polymer_system

{
    "repo_root": repo_root,
    "final_gro_path": final_gro_path,
    "final_gro_exists": final_gro_path.exists(),
    "docking_output_dir": docking_output_dir,
}

{'repo_root': PosixPath('/Users/k20098771/opt/iPHASimulator_v2'),
 'final_gro_path': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/benchmark/PHB4/gromacs/solvated_polymer/step7_production.gro'),
 'final_gro_exists': False,
 'docking_output_dir': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/docking_inputs/PHB4')}

## Export Polymer PDB

HADDOCK accepts PDB input. This cell converts the selected GROMACS `.gro` coordinate file into a polymer PDB in `examples/output/docking_inputs/{polymer_system}/`.

Run this only after the production GROMACS workflow has produced `step7_production.gro`.

In [ ]:
def infer_element(atom_name: str) -> str:
    letters = "".join(character for character in atom_name.strip() if character.isalpha())
    if not letters:
        return ""
    first = letters[0].upper()
    if len(letters) >= 2 and letters[:2].upper() in {"CL", "BR", "NA", "MG", "ZN", "FE", "CA"}:
        return letters[:2].title()
    return first


def gro_to_pdb(gro_path: Path, pdb_path: Path) -> Path:
    lines = gro_path.read_text().splitlines()
    if len(lines) < 3:
        raise ValueError(f"Invalid GRO file: {gro_path}")

    atom_count = int(lines[1].strip())
    atom_lines = lines[2:2 + atom_count]
    box_line = lines[2 + atom_count] if len(lines) > 2 + atom_count else ""

    pdb_path.parent.mkdir(parents=True, exist_ok=True)
    with pdb_path.open("w") as handle:
        box_values = [float(value) for value in box_line.split()[:3]]
        if len(box_values) == 3:
            a, b, c = [value * 10.0 for value in box_values]
            handle.write(f"CRYST1{a:9.3f}{b:9.3f}{c:9.3f}{90.0:7.2f}{90.0:7.2f}{90.0:7.2f} P 1           1\n")

        for index, line in enumerate(atom_lines, start=1):
            residue_number = int(line[0:5])
            residue_name = line[5:10].strip() or "PHA"
            atom_name = line[10:15].strip() or f"X{index}"
            atom_number = int(line[15:20])
            x = float(line[20:28]) * 10.0
            y = float(line[28:36]) * 10.0
            z = float(line[36:44]) * 10.0
            element = infer_element(atom_name)
            handle.write(
                f"ATOM  {atom_number:5d} {atom_name[:4]:>4s} {residue_name[:3]:>3s} A"
                f"{residue_number:4d}    {x:8.3f}{y:8.3f}{z:8.3f}"
                f"{1.00:6.2f}{0.00:6.2f}          {element:>2s}\n"
            )
        handle.write("TER\nEND\n")

    return pdb_path


polymer_pdb_path = docking_output_dir / f"{polymer_system}_step7_production_for_haddock.pdb"

if final_gro_path.exists():
    exported_polymer_pdb = gro_to_pdb(final_gro_path, polymer_pdb_path)
    print(f"Wrote {exported_polymer_pdb}")
else:
    print(f"Missing final MD structure: {final_gro_path}")
    print("Run the benchmark GROMACS production workflow before exporting docking PDBs.")

## Enzyme Information

| Enzyme | Length | Catalytic triad | PHO binding | PHB binding | Notes |
| --- | ---: | --- | --- | --- | --- |
| GK13 | 245 aa | Ser139 / Asp195 / His227 | binds PHO | does not bind PHB | Benchmark enzyme for PHO recognition |
| Anc45 | 244 aa | Ser138 / Asp194 / His226 | binds PHO | does not bind PHB | About 20x stronger than GK13 for PHO |

In [3]:
enzymes = [
    {
        "enzyme": "GK13",
        "length": "245 aa",
        "catalytic_triad": "Ser139 / Asp195 / His227",
        "binds": "PHO",
        "does_not_bind": "PHB",
    },
    {
        "enzyme": "Anc45",
        "length": "244 aa",
        "catalytic_triad": "Ser138 / Asp194 / His226",
        "binds": "PHO",
        "does_not_bind": "PHB",
        "note": "About 20x stronger than GK13 for PHO",
    },
]

enzymes

[{'enzyme': 'GK13',
  'length': '245 aa',
  'catalytic_triad': 'Ser139 / Asp195 / His227',
  'binds': 'PHO',
  'does_not_bind': 'PHB'},
 {'enzyme': 'Anc45',
  'length': '244 aa',
  'catalytic_triad': 'Ser138 / Asp194 / His226',
  'binds': 'PHO',
  'does_not_bind': 'PHB',
  'note': 'About 20x stronger than GK13 for PHO'}]

## Manual HADDOCK Job Records

Use the exported polymer PDB as the ligand input for manual HADDOCK setup. Add enzyme PDB paths after preparing the enzyme structures outside this notebook.

In [4]:
docking_jobs = [
    {
        "job_name": f"{enzyme['enzyme']}_{polymer_system}_manual_haddock",
        "enzyme": enzyme["enzyme"],
        "enzyme_pdb": None,
        "polymer_system": polymer_system,
        "polymer_pdb": polymer_pdb_path,
        "haddock_submission": "manual",
    }
    for enzyme in enzymes
]

docking_jobs

NameError: name 'polymer_pdb_path' is not defined

## HADDOCK Preparation Checklist

- Confirm `step7_production.gro` is the intended final production structure for the selected polymer.
- Inspect the exported polymer PDB before upload.
- Prepare enzyme PDB files for GK13 and Anc45 separately.
- In HADDOCK, upload enzyme as receptor and polymer as ligand.
- Record active/passive residues and run identifiers outside this notebook or in the `docking_jobs` table above.
- Do not start enzyme-polymer MD from these docking inputs until docking poses have been reviewed.